# 레이어 정규화: 훈련 안정화 - 레이어 정규화의 수학과 효과

- Tutorial ID: `adv-1-1`
- Tutorial: 레이어 정규화: 훈련 안정화
- Section ID: `adv-1-1-1`
- Section: 레이어 정규화의 수학과 효과


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("레이어 정규화 (Layer Normalization)")
print("=" * 60)

In [ ]:
# 1. 레이어 정규화 구현

In [ ]:
print("\n1. 레이어 정규화 기본 구현")
print("-" * 45)

def layer_norm(x, gamma=None, beta=None, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    if gamma is not None:
        x_norm = gamma * x_norm
    if beta is not None:
        x_norm = x_norm + beta
    return x_norm

d_model = 8
x = np.random.randn(3, d_model) * 10 + 5
print(f"입력 x (shape: {x.shape}):")
print(f"  평균: {np.round(np.mean(x, axis=-1), 2)}")
print(f"  표준편차: {np.round(np.std(x, axis=-1), 2)}")

x_ln = layer_norm(x)
print(f"\n레이어 정규화 후:")
print(f"  평균: {np.round(np.mean(x_ln, axis=-1), 6)}")
print(f"  표준편차: {np.round(np.std(x_ln, axis=-1), 6)}")
print(f"  → 평균 ≈ 0, 표준편차 ≈ 1!")

In [ ]:
# 2. RMSNorm (LLaMA 등)

In [ ]:
print("\n2. RMSNorm (LLaMA, Gemma 등에서 사용)")
print("-" * 45)

def rms_norm(x, gamma=None, eps=1e-5):
    rms = np.sqrt(np.mean(x**2, axis=-1, keepdims=True) + eps)
    x_norm = x / rms
    if gamma is not None:
        x_norm = gamma * x_norm
    return x_norm

x_test = np.array([3.0, 4.0, 0.0, 0.0])
x_ln_test = layer_norm(x_test)
x_rms_test = rms_norm(x_test)

print(f"입력: {x_test}")
print(f"LayerNorm: {np.round(x_ln_test, 4)}")
print(f"RMSNorm:   {np.round(x_rms_test, 4)}")
print("RMSNorm: 평균 빼기 생략 → 빠름, 정보 보존")

In [ ]:
# 3. Pre-LN vs Post-LN

In [ ]:
print("\n3. Pre-LN vs Post-LN 아키텍처")
print("-" * 45)
print("""
Post-LN (원래 트랜스포머):
  x → Attn → ADD → LN → FFN → ADD → LN
      ↑_________|           ↑_________|

Pre-LN (GPT-2+):
  x → LN → Attn → ADD → LN → FFN → ADD
                   ↑                  ↑
  ↑________________|  ↑_______________|

Pre-LN 장점: 그래디언트 직접 흐름, 더 안정적
""")

In [ ]:
# 4. 가중치로 접기 (Folding)

In [ ]:
print("4. 레이어 노름을 가중치로 접기")
print("-" * 45)

d = 4
gamma = np.array([1.5, 0.5, 2.0, 1.0])
W = np.random.randn(d, d)

W_folded = W @ np.diag(gamma)

x = np.random.randn(d)
x_norm = layer_norm(x)

result1 = W @ (gamma * x_norm)
result2 = W_folded @ x_norm

print(f"방법 1 (W @ (γ × x_norm)): {np.round(result1, 4)}")
print(f"방법 2 (W_folded @ x_norm): {np.round(result2, 4)}")
print(f"동일: {np.allclose(result1, result2)}")
print("→ 추론 시 γ를 W에 미리 곱해 연산 최적화!")